# Run new local models on best prompts

Notebook configurabil pentru testarea modelelor locale `Gemma` și `Mistral` prin Ollama, folosind prompturile best per task.

Produce fișiere JSON compatibile cu evaluatorul existent (`src/prompting/evaluate_models.py`).

Pași recomandați:
1. Rulează celulele de configurare.
2. Verifică datasetul și prompturile detectate.
3. Rulează întâi cu `LIMIT = 3`.
4. Dacă totul este OK, setează `LIMIT = None` și rulează complet.

In [1]:
from pathlib import Path
import json
import re
import time
import requests
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

from jinja2 import Environment, FileSystemLoader, StrictUndefined

## 1. Configurare generală

In [2]:
# ============================================================================
# CONFIGURARE PRINCIPALĂ
# ============================================================================

from pathlib import Path

# Path-ul repo-ului tău
BASE_DIR = Path(
    r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-"
)

# Setează aici datasetul pe care vrei să rulezi.
# Dacă ai varianta cu 180 conversații, schimbă doar numele fișierului de aici.
DATASET_PATH = BASE_DIR / "data" / "master_dataset_refined_180.json"

# Pentru test rapid:
LIMIT = 3

# Pentru rulare completă:
# LIMIT = None

TASKS_TO_RUN = ["intent", "final_status", "incongruities"]
# TASKS_TO_RUN = ["intent"]
# TASKS_TO_RUN = ["final_status"]
# TASKS_TO_RUN = ["incongruities"]

MODELS_TO_RUN = ["gemma3_4b", "mistral_7b"]
# MODELS_TO_RUN = ["gemma3_4b"]
# MODELS_TO_RUN = ["mistral_7b"]

OLLAMA_URL = "http://localhost:11434/api/generate"

# Modele locale descărcate în Ollama.
# Numele din ollama_name trebuie să coincidă exact cu ce apare la `ollama list`.
MODELS = {
    "gemma3_4b": {
        "ollama_name": "gemma3:4b",
        "display_name": "Gemma 3 4B",
    },
    "mistral_7b": {
        "ollama_name": "mistral:7b-instruct",
        "display_name": "Mistral 7B Instruct",
    },
}

# Best prompts alese pentru experimentul suplimentar.
# Aici am pus exact path-urile pe care mi le-ai dat.
BEST_PROMPTS = {
    "intent": {
        "lang": "en",
        "version": "v4",
        "candidates": [
            Path(
                r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\prompts\intent_extraction\en_few_shot_v4.jinja"
            ),
        ],
    },
    "final_status": {
        "lang": "ro",
        "version": "v4",
        "candidates": [
            Path(
                r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\prompts\final_status\ro_final_status_v4.jinja"
            ),
        ],
    },
    "incongruities": {
        "lang": "en",
        "version": "v2",
        "candidates": [
            Path(
                r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\prompts\incongruities\en_incongruities_v2.jinja"
            ),
        ],
    },
}

TASK_OUTPUT_DIRS = {
    "intent": BASE_DIR / "outputs" / "intent",
    "final_status": BASE_DIR / "outputs_final_status",
    "incongruities": BASE_DIR / "outputs_incongruities",
}

LABEL_DEFINITION_PATHS = {
    "intent": BASE_DIR / "configs" / "intent_definitions.json",
    "final_status": BASE_DIR / "configs" / "final_status_definitions.json",
    "incongruities": BASE_DIR / "configs" / "incongruities_definitions.json",
}

FEW_SHOT_EXAMPLES_PATHS = {
    "intent": BASE_DIR / "configs" / "few_shot_examples_intent.json",
    "final_status": BASE_DIR / "configs" / "few_shot_examples_final_status.json",
    "incongruities": BASE_DIR / "configs" / "few_shot_examples_incongruities.json",
}

# Pentru modele locale, ținem output-ul scurt ca să nu dureze mult.
OLLAMA_OPTIONS = {
    "temperature": 0.0,
    "num_predict": 96,
}

## 2. Verificare dataseturi și prompturi

In [3]:
def load_json(path: Path) -> Any:
    if not path.exists():
        raise FileNotFoundError(f"Nu există fișierul: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def count_conversations_in_file(path: Path) -> Optional[int]:
    try:
        data = load_json(path)
        if isinstance(data, list):
            return len(data)
        if isinstance(data, dict):
            for key in ["conversations", "data", "items", "dataset"]:
                if key in data and isinstance(data[key], list):
                    return len(data[key])
        return None
    except Exception:
        return None


print("BASE_DIR:", BASE_DIR)
print("DATASET_PATH:", DATASET_PATH)

processed_dir = BASE_DIR / "data" 
if processed_dir.exists():
    print("\nDataseturi JSON găsite în data/processed:")
    for p in sorted(processed_dir.glob("*.json")):
        n = count_conversations_in_file(p)
        print(f" - {p.name}: {n} conversații")
else:
    print("Nu există folderul data/processed.")

print("\nPrompturi selectate:")
for task, cfg in BEST_PROMPTS.items():
    existing = [p for p in cfg["candidates"] if p.exists()]
    chosen = existing[0] if existing else None
    print(f" - {task}: {cfg['lang']} {cfg['version']} -> {chosen}")

BASE_DIR: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-
DATASET_PATH: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\data\master_dataset_refined_180.json

Dataseturi JSON găsite în data/processed:
 - generated_conversations_180.json: 34 conversații
 - master_dataset_refined_180.json: 180 conversații

Prompturi selectate:
 - intent: en v4 -> C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\prompts\intent_extraction\en_few_shot_v4.jinja
 - final_status: ro v4 -> C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\prompts\final_status\ro_final_status_v4.jinja
 - incongruities: en v2 -> C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvi

## 3. Funcții de încărcare și normalizare

In [4]:
def resolve_prompt_path(task: str) -> Path:
    candidates = BEST_PROMPTS[task]["candidates"]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Nu am găsit niciun prompt pentru task={task}. Candidați:\n" +
        "\n".join(str(p) for p in candidates)
    )


def load_dataset(path: Path) -> List[Dict[str, Any]]:
    data = load_json(path)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ["conversations", "data", "items", "dataset"]:
            if key in data and isinstance(data[key], list):
                return data[key]
    raise ValueError("Nu pot identifica lista de conversații din dataset.")


def load_label_definitions(task: str) -> Any:
    path = LABEL_DEFINITION_PATHS[task]
    if not path.exists():
        return None
    return load_json(path)


def load_few_shot_examples(task: str) -> List[Dict[str, Any]]:
    path = FEW_SHOT_EXAMPLES_PATHS.get(task)
    if path is None or not path.exists():
        return []

    data = load_json(path)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ["examples", "few_shot_examples", "items", "data"]:
            if key in data and isinstance(data[key], list):
                return data[key]

    return []


def normalize_str(value: Any) -> Optional[str]:
    if value is None:
        return None
    return str(value).strip().lower().replace(" ", "_")


def get_ground_truth(conv: Dict[str, Any], task: str) -> Dict[str, Any]:
    if task == "intent":
        return {
            "dataset_label": conv.get("intent"),
            "dataset_intent": conv.get("intent"),
        }

    if task == "final_status":
        return {
            "dataset_status": conv.get("final_status"),
            "dataset_label": conv.get("final_status"),
        }

    if task == "incongruities":
        incongruities = conv.get("incongruities", [])
        has_inc = bool(incongruities)

        if has_inc:
            first = incongruities[0]
            if isinstance(first, dict):
                inc_type = first.get("type") or first.get("label")
            else:
                inc_type = str(first)
        else:
            inc_type = "none"

        return {
            "dataset_has_incongruity": has_inc,
            "dataset_type": inc_type,
            "dataset_label": inc_type,
        }

    raise ValueError(f"Task necunoscut: {task}")

## 4. Randarea promptului Jinja

In [5]:
def load_template(template_path: Path):
    env = Environment(
        loader=FileSystemLoader(str(template_path.parent)),
        undefined=StrictUndefined,
        trim_blocks=True,
        lstrip_blocks=True,
    )
    return env.get_template(template_path.name)


def output_format_instruction(task: str) -> str:
    if task == "intent":
        return '''
IMPORTANT OUTPUT FORMAT:
Return ONLY one valid JSON object.
Do not add explanations, markdown, code fences, or extra text.

Required format:
{"predicted_intent": "one_label_from_the_allowed_list"}
'''

    if task == "final_status":
        return '''
IMPORTANT OUTPUT FORMAT:
Return ONLY one valid JSON object.
Do not add explanations, markdown, code fences, or extra text.

Required format:
{"predicted_status": "one_label_from_the_allowed_list"}
'''

    return '''
IMPORTANT OUTPUT FORMAT:
Return ONLY one valid JSON object.
Do not add explanations, markdown, code fences, or extra text.

Required format:
{
  "predicted_has_incongruity": true,
  "predicted_type": "one_label_from_the_allowed_list"
}

If there is no incongruity, use:
{
  "predicted_has_incongruity": false,
  "predicted_type": "none"
}
'''


def render_prompt(task: str, conv: Dict[str, Any]) -> str:
    template_path = resolve_prompt_path(task)
    template = load_template(template_path)

    # IMPORTANT:
    # Template-urile tale fac de obicei:
    # {% for turn in conversation %}
    # deci conversation trebuie să fie listă de turn-uri, nu string.
    context = {
        "conversation": conv.get("turns", []),
        "conversation_id": conv.get("conversation_id") or conv.get("id"),
        "turns": conv.get("turns", []),
        "intent": conv.get("intent"),
        "final_status": conv.get("final_status"),
        "incongruities": conv.get("incongruities", []),
        "label_definitions": load_label_definitions(task),
        "examples": load_few_shot_examples(task),
    }

    prompt = template.render(**context)

    # Wrapper util pentru modele locale, ca să reducem parse_fail.
    return prompt + "\n\n" + output_format_instruction(task)

In [8]:
# ============================================================================
# AFIȘARE FULL PROMPTS RANDATE PENTRU TOATE CELE 3 TASKURI
# ============================================================================

# Încarcă datasetul dacă nu există deja
if "dataset" not in globals():
    dataset = load_dataset(DATASET_PATH)
    print("Dataset încărcat automat:", DATASET_PATH)
    print("Total conversații:", len(dataset))

# Alege conversația pentru care vrei să vezi prompturile
CONV_INDEX_FOR_PREVIEW = 0

for task in ["intent", "final_status", "incongruities"]:
    template_path = resolve_prompt_path(task)
    examples = load_few_shot_examples(task)

    conv = dataset[CONV_INDEX_FOR_PREVIEW]
    conv_id = conv.get("conversation_id") or conv.get("id") or f"conv_{CONV_INDEX_FOR_PREVIEW:04d}"

    prompt = render_prompt(task, conv)

    print("\n\n")
    print("#" * 120)
    print(f"FULL RENDERED PROMPT | TASK: {task}")
    print("#" * 120)
    print(f"TEMPLATE: {template_path}")
    print(f"CONVERSATION ID: {conv_id}")
    print(f"FEW-SHOT EXAMPLES LOADED: {len(examples)}")
    print(f"PROMPT LENGTH: {len(prompt)} caractere")
    print("#" * 120)
    print(prompt)
    print("#" * 120)




########################################################################################################################
FULL RENDERED PROMPT | TASK: intent
########################################################################################################################
TEMPLATE: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\prompts\intent_extraction\en_few_shot_v4.jinja
CONVERSATION ID: conv_simple_0001
FEW-SHOT EXAMPLES LOADED: 6
PROMPT LENGTH: 7386 caractere
########################################################################################################################
Developer: # Role and Objective
You are an intent classification assistant for a Romanian banking voicebot system.

Your objective is to identify the **primary intent** of the entire conversation below. The intent is determined by what the user wanted to accomplish within the conversation.

# Intent Labels
- **`block_card`*

## 5. Apel Ollama și parsare JSON

In [9]:
def call_ollama(model_name: str, prompt: str) -> Tuple[str, float]:
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": OLLAMA_OPTIONS,
    }

    start = time.perf_counter()
    response = requests.post(OLLAMA_URL, json=payload, timeout=240)
    latency_ms = (time.perf_counter() - start) * 1000

    response.raise_for_status()
    data = response.json()

    return data.get("response", ""), latency_ms


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = text.strip()
    text = text.replace("```json", "").replace("```JSON", "").replace("```", "").strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if not match:
        return None

    candidate = match.group(0).strip()

    for candidate_variant in [candidate, candidate.replace("'", '"')]:
        try:
            parsed = json.loads(candidate_variant)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass

    return None


def parse_prediction(task: str, raw_text: str) -> Dict[str, Any]:
    parsed = extract_json_object(raw_text)

    if parsed is None:
        return {
            "parse_failed": True,
            "raw_response": raw_text,
        }

    result = {
        "parse_failed": False,
        "raw_response": raw_text,
        "parsed_response": parsed,
    }

    if task == "intent":
        pred = (
            parsed.get("predicted_intent")
            or parsed.get("intent")
            or parsed.get("label")
            or parsed.get("prediction")
        )
        result["predicted_intent"] = normalize_str(pred)
        result["predicted_label"] = normalize_str(pred)

    elif task == "final_status":
        pred = (
            parsed.get("predicted_status")
            or parsed.get("predicted_final_status")
            or parsed.get("final_status")
            or parsed.get("status")
            or parsed.get("label")
            or parsed.get("prediction")
        )
        result["predicted_status"] = normalize_str(pred)
        result["predicted_label"] = normalize_str(pred)

    elif task == "incongruities":
        has_inc = (
            parsed.get("predicted_has_incongruity")
            if "predicted_has_incongruity" in parsed
            else parsed.get("has_incongruity")
        )

        pred_type = (
            parsed.get("predicted_type")
            or parsed.get("predicted_incongruity_type")
            or parsed.get("type")
            or parsed.get("label")
            or parsed.get("prediction")
        )

        pred_type_norm = normalize_str(pred_type)

        if isinstance(has_inc, str):
            has_inc_norm = has_inc.strip().lower() in {"true", "yes", "da", "1", "exista", "există"}
        elif has_inc is None:
            has_inc_norm = pred_type_norm not in {None, "none", "nu", "false", "fara_neconcordanta"}
        else:
            has_inc_norm = bool(has_inc)

        result["predicted_has_incongruity"] = has_inc_norm
        result["predicted_type"] = pred_type_norm if has_inc_norm else "none"
        result["predicted_label"] = result["predicted_type"]

    return result

## 6. Teste rapide înainte de rulare

In [10]:
# Verifică dacă Ollama răspunde.
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=10)
    print("Ollama status:", r.status_code)
    print("Modele disponibile:")
    for m in r.json().get("models", []):
        print(" -", m.get("name"))
except Exception as e:
    print("Nu pot contacta Ollama. Verifică dacă Ollama rulează.")
    print("Eroare:", e)


# Încarcă datasetul.
dataset = load_dataset(DATASET_PATH)
print("\nDataset încărcat:", DATASET_PATH)
print("Total conversații:", len(dataset))

# Verifică exemplele few-shot.
for task in TASKS_TO_RUN:
    print(f"{task}: {len(load_few_shot_examples(task))} few-shot examples")

Ollama status: 200
Modele disponibile:
 - mistral:7b-instruct
 - gemma3:4b
 - aya-expanse:8b

Dataset încărcat: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\data\master_dataset_refined_180.json
Total conversații: 180
intent: 6 few-shot examples
final_status: 7 few-shot examples
incongruities: 7 few-shot examples


In [11]:
# Randare prompt pentru primul exemplu, fără a chema modelul.
task = TASKS_TO_RUN[0]
sample_prompt = render_prompt(task, dataset[0])
print("Task:", task)
print("Lungime prompt:", len(sample_prompt), "caractere")
print("\nPreview prompt:\n")
print(sample_prompt[:2500])

Task: intent
Lungime prompt: 7386 caractere

Preview prompt:

Developer: # Role and Objective
You are an intent classification assistant for a Romanian banking voicebot system.

Your objective is to identify the **primary intent** of the entire conversation below. The intent is determined by what the user wanted to accomplish within the conversation.

# Intent Labels
- **`block_card`**: The user wants to block a lost, stolen, retained, or compromised card.
- **`unblock_card`**: The user wants to unblock a previously blocked card.
- **`open_account`**: The user wants to open a new bank account.
- **`close_account`**: The user wants to close an existing bank account.
- **`check_balance`**: The user wants to check the account balance or available funds.
- **`get_account_statement`**: The user wants to obtain the account statement or transaction history.
- **`report_suspicious_transaction`**: The user reports a suspicious, fraudulent, or unauthorized transaction.
- **`update_personal_data`

## 7. Rulare experiment

In [12]:
def run_experiment(
    task: str,
    model_key: str,
    model_cfg: Dict[str, str],
    dataset: List[Dict[str, Any]],
    limit: Optional[int] = None,
) -> Path:
    prompt_cfg = BEST_PROMPTS[task]
    template_path = resolve_prompt_path(task)

    output_dir = TASK_OUTPUT_DIRS[task]
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    lang = prompt_cfg["lang"]
    version = prompt_cfg["version"]

    if task == "intent":
        prefix = "exp"
    elif task == "final_status":
        prefix = "exp_fst"
    else:
        prefix = "exp_inc"

    experiment_name = f"{prefix}_{model_key}__{lang}__{version}"

    predictions = []
    subset = dataset[:limit] if limit else dataset

    print(f"\n=== {task.upper()} | {model_key} | {lang} {version} | N={len(subset)} ===")

    for idx, conv in enumerate(subset, start=1):
        conv_id = conv.get("conversation_id") or conv.get("id") or f"conv_{idx:04d}"

        try:
            prompt = render_prompt(task, conv)
            raw_response, latency_ms = call_ollama(model_cfg["ollama_name"], prompt)
            pred = parse_prediction(task, raw_response)

        except Exception as e:
            latency_ms = None
            pred = {
                "parse_failed": True,
                "error": str(e),
                "raw_response": "",
            }

        row = {
            "conversation_id": conv_id,
            "model": model_key,
            "prompt_lang": lang,
            "prompt_version": version,
            "latency_ms": latency_ms,
            **get_ground_truth(conv, task),
            **pred,
        }

        predictions.append(row)

        status = "OK" if not row.get("parse_failed") else "PARSE_FAIL"
        latency_txt = f"{latency_ms/1000:.1f}s" if latency_ms is not None else "-"
        print(f"[{idx:04d}/{len(subset)}] {conv_id} -> {status} | {latency_txt}")

    output = {
        "experiment_name": experiment_name,
        "model": model_key,
        "model_display": model_cfg["display_name"],
        "provider": "Ollama local",
        "ollama_model": model_cfg["ollama_name"],
        "language": lang,
        "prompt_version": version,
        "task": task,
        "timestamp": timestamp,
        "prompt_template": str(template_path),
        "n_predictions": len(predictions),
        "results": predictions,
        "predictions": predictions,
    }

    out_path = output_dir / f"{experiment_name}__{timestamp}.json"

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    n_fail = sum(1 for r in predictions if r.get("parse_failed"))
    print(f"Saved: {out_path}")
    print(f"Parse failures: {n_fail}/{len(predictions)}")

    return out_path

In [ ]:
generated_files = []

for task in TASKS_TO_RUN:
    for model_key in MODELS_TO_RUN:
        if model_key not in MODELS:
            raise ValueError(f"Model necunoscut: {model_key}. Disponibile: {list(MODELS.keys())}")

        out_path = run_experiment(
            task=task,
            model_key=model_key,
            model_cfg=MODELS[model_key],
            dataset=dataset,
            limit=LIMIT,
        )
        generated_files.append(out_path)

print("\nFișiere generate:")
for p in generated_files:
    print(" -", p)


=== INTENT | gemma3_4b | en v4 | N=3 ===
[0001/3] conv_simple_0001 -> OK | 115.8s
[0002/3] conv_simple_0002 -> OK | 115.6s


## 8. După ce rularea completă este gata

In [ ]:
print("După ce ai rulat cu LIMIT = None, rulează în terminal:")
print("python src/prompting/evaluate_models.py --task all --save-latex")

In [1]:
# ============================================================================
# METRICI RAPIDE PENTRU EXPERIMENTELE NOI — QWEN + MISTRAL
# ============================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score


# ============================================================================
# 1. CĂI OUTPUT
# ============================================================================

TASK_OUTPUT_DIRS = {
    "intent": Path(
        r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs\intent"
    ),
    "final_status": Path(
        r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_final_status"
    ),
    "incongruities": Path(
        r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities"
    ),
}

NEW_MODELS = [
    "qwen2_5_3b",
    "mistral_7b",
]


# ============================================================================
# 2. FUNCȚII UTILITARE
# ============================================================================

def load_experiment(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_predictions_container(exp):
    if isinstance(exp.get("results"), list):
        return exp["results"]
    if isinstance(exp.get("predictions"), list):
        return exp["predictions"]
    return []


def pick_first(row, keys):
    for k in keys:
        if k in row and row[k] is not None:
            return row[k]
    return None


def norm(x):
    if x is None:
        return None
    return str(x).strip().lower().replace(" ", "_")


def extract_y_true_y_pred(task, rows):
    y_true = []
    y_pred = []
    parse_failed = 0
    latencies = []

    for r in rows:
        if r.get("latency_ms") is not None:
            latencies.append(r["latency_ms"])

        if r.get("parse_failed"):
            parse_failed += 1
            continue

        if task == "intent":
            true = pick_first(
                r,
                ["dataset_intent", "dataset_label", "ground_truth", "true_label", "label"]
            )
            pred = pick_first(
                r,
                ["predicted_intent", "predicted_label", "prediction", "pred_label"]
            )

        elif task == "final_status":
            true = pick_first(
                r,
                ["dataset_status", "dataset_label", "ground_truth", "true_label", "label"]
            )
            pred = pick_first(
                r,
                [
                    "predicted_status",
                    "predicted_final_status",
                    "predicted_label",
                    "prediction",
                    "pred_label",
                ]
            )

        elif task == "incongruities":
            true = pick_first(
                r,
                [
                    "dataset_type",
                    "dataset_incongruity_type",
                    "dataset_label",
                    "ground_truth",
                    "true_label",
                    "label",
                ]
            )
            pred = pick_first(
                r,
                [
                    "predicted_type",
                    "predicted_incongruity_type",
                    "predicted_label",
                    "prediction",
                    "pred_label",
                ]
            )

            # Normalizează cazul fără neconcordanță
            if norm(true) in ["", "null", "none", "false"]:
                true = "none"
            if norm(pred) in ["", "null", "none", "false"]:
                pred = "none"

        else:
            raise ValueError(f"Task necunoscut: {task}")

        true = norm(true)
        pred = norm(pred)

        if true is None or pred is None:
            parse_failed += 1
            continue

        y_true.append(true)
        y_pred.append(pred)

    return y_true, y_pred, parse_failed, latencies


def compute_metrics(y_true, y_pred):
    if not y_true:
        return {
            "accuracy": np.nan,
            "macro_f1": np.nan,
            "weighted_f1": np.nan,
            "kappa": np.nan,
        }

    labels = sorted(set(y_true) | set(y_pred))

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="weighted",
            zero_division=0,
        ),
        "kappa": cohen_kappa_score(y_true, y_pred, labels=labels),
    }


# ============================================================================
# 3. PARCURGERE FIȘIERE ȘI CALCUL METRICI
# ============================================================================

summary_rows = []

for task, out_dir in TASK_OUTPUT_DIRS.items():
    if not out_dir.exists():
        print(f"Folder lipsă pentru {task}: {out_dir}")
        continue

    json_files = sorted(out_dir.glob("*.json"))
    print(f"{task}: {len(json_files)} fișiere JSON găsite în {out_dir}")

    for path in json_files:
        try:
            exp = load_experiment(path)
        except Exception as e:
            print(f"Nu pot citi {path}: {e}")
            continue

        model = exp.get("model", "")
        experiment_name = exp.get("experiment_name", path.stem)

        # Păstrează doar Qwen și Mistral
        if model not in NEW_MODELS and not any(m in experiment_name for m in NEW_MODELS):
            continue

        rows = get_predictions_container(exp)

        y_true, y_pred, parse_failed, latencies = extract_y_true_y_pred(task, rows)
        metrics = compute_metrics(y_true, y_pred)

        n_total = len(rows)
        n_valid = len(y_true)

        summary_rows.append({
            "task": task,
            "experiment": experiment_name,
            "model": model,
            "language": exp.get("language"),
            "prompt_version": exp.get("prompt_version"),
            "n_total": n_total,
            "n_valid": n_valid,
            "parse_failed": parse_failed,
            "parse_fail_rate_%": round(parse_failed / n_total * 100, 2) if n_total else np.nan,
            "accuracy_%": round(metrics["accuracy"] * 100, 2) if not np.isnan(metrics["accuracy"]) else np.nan,
            "macro_f1_%": round(metrics["macro_f1"] * 100, 2) if not np.isnan(metrics["macro_f1"]) else np.nan,
            "weighted_f1_%": round(metrics["weighted_f1"] * 100, 2) if not np.isnan(metrics["weighted_f1"]) else np.nan,
            "kappa": round(metrics["kappa"], 4) if not np.isnan(metrics["kappa"]) else np.nan,
            "latency_mean_s": round(np.mean(latencies) / 1000, 2) if latencies else np.nan,
            "latency_median_s": round(np.median(latencies) / 1000, 2) if latencies else np.nan,
            "file": str(path),
        })


# ============================================================================
# 4. AFIȘARE REZULTATE
# ============================================================================

df_metrics_new = pd.DataFrame(summary_rows)

if df_metrics_new.empty:
    print("Nu am găsit experimente noi pentru modelele:", NEW_MODELS)
else:
    df_metrics_new = df_metrics_new.sort_values(
        ["task", "macro_f1_%"],
        ascending=[True, False],
    )

    display(df_metrics_new[[
        "task",
        "model",
        "language",
        "prompt_version",
        "n_total",
        "n_valid",
        "parse_failed",
        "parse_fail_rate_%",
        "accuracy_%",
        "macro_f1_%",
        "weighted_f1_%",
        "kappa",
        "latency_mean_s",
        "latency_median_s",
    ]])


# ============================================================================
# 5. BEST PER TASK
# ============================================================================

if not df_metrics_new.empty:
    best_new = (
        df_metrics_new
        .sort_values(["task", "macro_f1_%"], ascending=[True, False])
        .groupby("task")
        .head(1)
    )

    print("\nBEST PER TASK:")
    display(best_new[[
        "task",
        "model",
        "language",
        "prompt_version",
        "accuracy_%",
        "macro_f1_%",
        "weighted_f1_%",
        "kappa",
        "parse_fail_rate_%",
        "latency_mean_s",
    ]])


# ============================================================================
# 6. OPȚIONAL: SALVARE CSV
# ============================================================================

if not df_metrics_new.empty:
    csv_path = Path(
        r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\evaluation_reports\new_models_quick_metrics.csv"
    )
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_metrics_new.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"\nCSV salvat la: {csv_path}")

intent: 44 fișiere JSON găsite în C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs\intent
final_status: 42 fișiere JSON găsite în C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_final_status
incongruities: 42 fișiere JSON găsite în C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_incongruities


,task,model,language,prompt_version,n_total,n_valid,parse_failed,parse_fail_rate_%,accuracy_%,macro_f1_%,weighted_f1_%,kappa,latency_mean_s,latency_median_s
2,final_status,mistral_7b,ro,v4,180,178,2,1.11,69.66,55.06,67.04,0.5463,22.72,22.19
3,final_status,qwen2_5_3b,ro,v4,180,174,6,3.33,52.87,49.88,55.20,0.3842,13.93,13.84
4,incongruities,mistral_7b,en,v2,180,179,1,0.56,79.33,21.33,73.31,0.2195,8.93,8.86
5,incongruities,qwen2_5_3b,en,v2,180,180,0,0.00,75.56,15.69,72.92,0.2562,6.79,6.59
0,intent,mistral_7b,en,v4,180,180,0,0.00,89.44,81.91,88.55,0.8849,9.31,8.98
1,intent,qwen2_5_3b,en,v4,180,180,0,0.00,75.00,69.67,70.25,0.7274,6.04,6.05



BEST PER TASK:


,task,model,language,prompt_version,accuracy_%,macro_f1_%,weighted_f1_%,kappa,parse_fail_rate_%,latency_mean_s
2,final_status,mistral_7b,ro,v4,69.66,55.06,67.04,0.5463,1.11,22.72
4,incongruities,mistral_7b,en,v2,79.33,21.33,73.31,0.2195,0.56,8.93
0,intent,mistral_7b,en,v4,89.44,81.91,88.55,0.8849,0.00,9.31



CSV salvat la: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\evaluation_reports\new_models_quick_metrics.csv


In [ ]:
from pathlib import Path

BASE_DIR = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-")

for task in ["intent", "final_status", "incongruities"]:
    txt_file = BASE_DIR / "evaluation_reports" / f"eval_{task}.txt"
    if not txt_file.exists():
        continue
    
    print(f"\n{'='*100}")
    print(f"  {task.upper()} — Tabel T2 (toate experimentele)")
    print(f"{'='*100}")
    
    with open(txt_file, encoding="utf-8") as f:
        content = f.read()
    
    # Extrage secțiunea T2 (All experiments ranked)
    if "T2 —" in content or "All experiments ranked" in content:
        # Găsește începutul tabelului T2
        start = content.find("T2 —")
        if start == -1:
            start = content.find("All experiments ranked")
        
        # Găsește sfârșitul (următorul tabel sau sfârșitul fișierului)
        end = content.find("T3 —", start)
        if end == -1:
            end = content.find("Per-class", start)
        if end == -1:
            end = len(content)
        
        section = content[start:end]
        
        # Afișează doar liniile cu Qwen/Mistral
        for line in section.split('\n'):
            if any(x in line for x in ["Qwen", "Mistral", "Model", "│", "├", "╰", "╭"]):
                print(line)

UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 69: character maps to <undefined>